# Ch. 3 · TP1 Régression scientifique

**IA pour les sciences — M1.** Comparaison à une baseline et validation croisée.

Exécutez les cellules dans l’ordre. La démarche et ses limites sont discutées dans le chapitre HTML correspondant. Les données artificielles utilisées, le cas échéant, servent uniquement à l’enseignement.

## Préparation de l’environnement

Dépendances usuelles : `numpy`, `pandas`, `matplotlib`, `scikit-learn` (et `scipy` pour le chapitre 6). Sur Colab, elles sont généralement disponibles.

### Étape 1

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)
n = 180
T = rng.uniform(10, 75, n)
y = 1.5 + 0.8*T + 0.015*T**2 + rng.normal(0, 7, n)
df = pd.DataFrame({'temperature_C': T, 'signal_u.a.': y})
print(df.describe())
df.plot.scatter(x='temperature_C', y='signal_u.a.')
plt.show()

### Étape 2

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

X = df[['temperature_C']]
y = df['signal_u.a.']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)
models = {
    'Référence': DummyRegressor(strategy='mean'),
    'Linéaire': LinearRegression(),
    'Forêt': RandomForestRegressor(n_estimators=100,
                                   min_samples_leaf=3, random_state=42)
}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(name, 'MAE =', round(mean_absolute_error(y_test, pred), 2),
          'RMSE =', round(root_mean_squared_error(y_test, pred), 2))

### Étape 3

In [ ]:
scores = cross_val_score(models['Forêt'], X_train, y_train,
                          cv=5, scoring='neg_mean_absolute_error')
print('MAE CV :', -scores.mean(), '±', scores.std())
residus = y_test - models['Forêt'].predict(X_test)
plt.scatter(models['Forêt'].predict(X_test), residus)
plt.axhline(0, color='black', linestyle='--')
plt.xlabel('Prédiction (u.a.)'); plt.ylabel('Résidu (u.a.)')
plt.show()

## Interprétation scientifique

1. Décrivez les données et leurs limites.
2. Justifiez la méthode et les métriques.
3. Discutez ce que les résultats permettent de conclure et ce qu’ils **ne permettent pas** de conclure.
4. Proposez au moins un contrôle de robustesse ou une expérience complémentaire.